# Импорт библиотек

In [2]:
import sys
import os

sys.path.append(os.path.abspath('lib'))

from preprocessing_pipeline import create_combined_pipeline
from test_models import run_models_classifications

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

# Считывание данных

In [4]:
df = pd.read_csv('data/processed.csv')
df.shape

(1001, 213)

# Удаление невалидных экземляров

In [6]:
matching_instances = df[df['IC50, mM'] == df['CC50, mM']]
print(len(matching_instances.index))
df.drop(index=matching_instances.index, inplace=True)

135


# Очистка таргета от выбросов

In [8]:
df = df[df['IC50, mM'] < 4000]

# Подготовка данных для эксперемента

In [10]:
combined_pipeline = create_combined_pipeline()
X = df.drop(columns=['IC50, mM', 'CC50, mM', 'SI'])
y1 = df['IC50, mM']
y2 = df['CC50, mM']
y3 = df['SI']
X_transformed = combined_pipeline.fit_transform(X)
df = pd.concat([X_transformed, y1, y2, y3], axis=1)

приоритет на recall, чтобы не упустить хорошее лекарство

In [12]:
X = df.drop(columns=['IC50, mM', 'CC50, mM', 'SI'])
y = df['IC50, mM'].apply(lambda v: 1 if v >= df['IC50, mM'].median() else 0)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Train dataset size: {X_train.shape}, {y_train.shape}')
print(f'Test dataset size: {X_test.shape}, {y_test.shape}')

Train dataset size: (692, 98), (692,)
Test dataset size: (174, 98), (174,)


# Эксперемент с моделями

In [14]:
run_models_classifications(X_train, X_test, y_train, y_test)

,Model,WA f1,accuracy,WA precision,WA recall,WA support,roc_auc
4,XGBoost,0.74,0.74,0.75,0.74,174.0,0.78
3,Random Forest,0.73,0.72,0.75,0.72,174.0,0.77
7,CatBoost,0.73,0.72,0.74,0.72,174.0,0.79
9,AdaBoost,0.73,0.72,0.74,0.72,174.0,0.77
6,LightGBM,0.71,0.71,0.73,0.71,174.0,0.80
1,Decision Tree,0.70,0.70,0.70,0.70,174.0,0.69
8,HistGradientBoosting,0.69,0.69,0.71,0.69,174.0,0.78
5,Gradient Boosting,0.67,0.67,0.69,0.67,174.0,0.77
0,Logistic Regression,0.65,0.64,0.66,0.64,174.0,0.70
2,KNeighbors,0.62,0.63,0.62,0.63,174.0,0.67


# Подбор гиперпараметров

In [16]:
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import classification_report, roc_auc_score
from scipy.stats import randint, uniform

param_dist = {
    'iterations': randint(50, 300),
    'depth': randint(3, 10),
    'learning_rate': uniform(0.01, 0.3),
    'l2_leaf_reg': uniform(1, 10),
    'border_count': randint(32, 255),
    'grow_policy': ['SymmetricTree', 'Depthwise', 'Lossguide'],
    'random_strength': uniform(0, 1),
    'leaf_estimation_iterations': randint(1, 10),
    'bootstrap_type': ['Bayesian', 'Bernoulli', 'MVS'],
    'feature_border_type': ['GreedyLogSum', 'Median', 'Uniform'],
}
random_search = RandomizedSearchCV(
    estimator=CatBoostClassifier(silent=True, random_state=42),
    param_distributions=param_dist,
    n_iter=200,
    scoring='roc_auc',
    cv=5,
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

best_params = random_search.best_params_
print("Лучшие параметры:", best_params)

/Users/v.papadyk/anaconda3/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Лучшие параметры: {'bootstrap_type': 'Bayesian', 'border_count': 47, 'depth': 6, 'feature_border_type': 'Uniform', 'grow_policy': 'SymmetricTree', 'iterations': 59, 'l2_leaf_reg': 1.9169933511143014, 'leaf_estimation_iterations': 7, 'learning_rate': 0.16675575749035693, 'random_strength': 0.7322718071846451}


In [17]:
model = CatBoostClassifier(
    bootstrap_type='Bayesian',
    border_count=47,
    depth=6,
    feature_border_type='Uniform',
    grow_policy='SymmetricTree',
    iterations=59,
    l2_leaf_reg=1.92,
    leaf_estimation_iterations=7,
    learning_rate=0.17,
    random_strength=0.73,
    random_state=42,
    verbose=0,
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_score = model.predict_proba(X_test)[:, 1]

report = classification_report(y_test, y_pred, output_dict=True)
report_df = pd.DataFrame(report).transpose()
roc_auc = roc_auc_score(y_test, y_score)

print("Accuracy:", round(report['accuracy'], 2))
print("ROC AUC Score:", round(roc_auc, 2))
report_df

Accuracy: 0.71
ROC AUC Score: 0.78


,precision,recall,f1-score,support
0,0.809524,0.666667,0.731183,102.000000
1,0.622222,0.777778,0.691358,72.000000
accuracy,0.712644,0.712644,0.712644,0.712644
macro avg,0.715873,0.722222,0.711270,174.000000
weighted avg,0.732020,0.712644,0.714704,174.000000
